# Transformation des fichiers Excel historiques vers les tables Polypbase

Notebook corrigé après la réunion métier :

- les additions dans les cellules de comptage sont converties en totaux ;
- les cellules bleues sont exportées comme absences de suivi justifiées, notamment Covid ;
- les évolutions de noms d'espèces sont gardées comme cas de validation métier, pas comme erreurs bloquantes ;
- les anomalies réellement bloquantes restent exportées à part et exclues des CSV finaux.


In [259]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from openpyxl import load_workbook

# Dossier du projet.
# Ce chemin doit pointer vers le clone propre du projet, pas vers la corbeille.
PROJECT_DIR = Path("/Users/akkouh/Desktop/POLYPBASE")

# On cherche les fichiers Excel dans plusieurs emplacements possibles.
# Le premier dossier contenant des fichiers Suivi_*.xlsx sera utilisé.
data_dir_candidates = [
    PROJECT_DIR / "data",
    PROJECT_DIR,
    Path("/Users/akkouh/Desktop/polybase"),
]

DATA_DIR = None
excel_files = []

for candidate in data_dir_candidates:
    files = sorted(candidate.glob("Suivi_*.xlsx")) if candidate.exists() else []
    if files:
        DATA_DIR = candidate
        excel_files = files
        break

# Si aucun fichier Excel n'est trouvé, on garde data/ par défaut.
if DATA_DIR is None:
    DATA_DIR = PROJECT_DIR / "data"
    excel_files = []

# Dossiers de sortie pour les tables propres, les anomalies et les fichiers de contrôle.
OUTPUT_DIR = PROJECT_DIR / "exports_polypbase"
ANOMALY_DIR = OUTPUT_DIR / "anomalies"
CONTROL_DIR = OUTPUT_DIR / "controles"

OUTPUT_DIR.mkdir(exist_ok=True)
ANOMALY_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier projet :", PROJECT_DIR)
print("Dossier données :", DATA_DIR)
print("Nombre de fichiers Excel trouvés :", len(excel_files))
display([file.name for file in excel_files])


Dossier projet : /Users/akkouh/Desktop/POLYPBASE
Dossier données : /Users/akkouh/Desktop/polybase
Nombre de fichiers Excel trouvés : 8


['Suivi_2019.xlsx',
 'Suivi_2020.xlsx',
 'Suivi_2021.xlsx',
 'Suivi_2022.xlsx',
 'Suivi_2023.xlsx',
 'Suivi_2024.xlsx',
 'Suivi_2025.xlsx',
 'Suivi_2026.xlsx']

In [260]:
# Vérification rapide des feuilles disponibles dans chaque fichier.

for file in excel_files:
    xls = pd.ExcelFile(file)
    print(f"\n{file.name}")
    print(xls.sheet_names)



Suivi_2019.xlsx
['Aurelia', 'Hydrozoa', 'Cubozoa', 'Rhizostomae', 'Semaestomeae', 'Coronatae']

Suivi_2020.xlsx
['Aurelia', 'Cubozoa', 'Hydrozoa', 'Rhizostomae', 'Semaestomeae', 'Coronatae']

Suivi_2021.xlsx
['Aurelia', 'Cubozoa', 'Hydrozoa', 'Rhizostomae', 'Semaestomeae', 'Coronatae']

Suivi_2022.xlsx
['Aurelia', 'Cubozoa', 'Hydrozoa', 'Rhizostomae', 'Semaestomeae', 'Coronatae']

Suivi_2023.xlsx
['Aurelia', 'Hydrozoa', 'Cubozoa', 'Rhizostomeae', 'Semaestomeae', 'Coronatae', 'UMR_MARBEC']

Suivi_2024.xlsx
['Aurelia', 'Hydrozoa', 'Cubozoa', 'Rhizostomae', 'Semaestomeae', 'Feuil1', 'Feuil2', 'Coronatae', 'UMR_MARBEC']

Suivi_2025.xlsx
['Aurelia', 'Hydrozoa', 'Cubozoa', 'Rhizostomae', 'Feuil1', 'Semaestomeae', 'Coronatae', 'Staurozoa', 'UMR_MARBEC']

Suivi_2026.xlsx
['Aurelia', 'Hydrozoa', 'Cubozoa', 'Rhizostomae', 'Feuil1', 'Semaestomeae', 'Coronatae']


In [261]:
def extract_year_from_filename(path: Path) -> int | None:
    # Extrait l'année depuis un nom de fichier du type Suivi_2025.xlsx.
    match = re.search(r"(\d{4})", path.name)
    return int(match.group(1)) if match else None


def clean_text(value):
    # Nettoie une cellule texte et retourne None si elle est vide.
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value else None


def normalize_measure_type(value):
    # Normalise le type de mesure biologique.
    value = clean_text(value)
    if value is None:
        return None

    value_lower = value.lower()

    if "polype" in value_lower:
        return "polypes"
    if "éphyr" in value_lower or "ephyr" in value_lower:
        return "ephyrules"

    return value_lower


def get_cell_rgb(cell) -> str | None:
    """Retourne la couleur RGB d'une cellule Excel si elle existe."""
    fill = cell.fill
    if fill is None or fill.fill_type is None:
        return None

    color = fill.fgColor
    if color is None:
        return None

    if color.type == "rgb" and color.rgb:
        return color.rgb[-6:].upper()

    return None


def classify_cell_fill(cell) -> str:
    """Classe les couleurs utiles dans les fichiers Excel historiques.

    - bleu : absence de suivi / période Covid ou période particulière sans comptage fiable ;
    - gris : boîte inexistante, pas encore créée ou supprimée ;
    - standard : cellule normale.
    """
    rgb = get_cell_rgb(cell)
    if rgb is None:
        return "standard"

    r = int(rgb[0:2], 16)
    g = int(rgb[2:4], 16)
    b = int(rgb[4:6], 16)

    # Bleu clair ou bleu soutenu : dans les fichiers, cela matérialise une absence de suivi.
    if b >= 140 and g >= 120 and r <= 190 and b > r + 20:
        return "absence_suivi"

    # Gris : les trois composantes sont proches.
    if abs(r - g) <= 12 and abs(g - b) <= 12 and 120 <= r <= 230:
        return "boite_inexistante"

    return "standard"


def nettoyer_valeur_releve(valeur):
    """Nettoie une valeur de comptage.

    Les additions écrites en texte dans Excel sont interprétées comme des totaux.
    Exemple : 5+4 devient 9, 200+200 devient 400.
    """
    if pd.isna(valeur):
        return pd.NA

    texte = str(valeur).strip()
    if texte == "":
        return pd.NA

    # Cas des additions écrites directement dans une cellule : 5+4, 2 + 2 + 1, etc.
    if re.fullmatch(r"\d+(\s*\+\s*\d+)+", texte):
        return sum(int(part.strip()) for part in texte.split("+"))

    try:
        nombre = float(texte.replace(",", "."))
        if nombre.is_integer():
            return int(nombre)
        return nombre
    except ValueError:
        return pd.NA


def est_addition_releve(valeur) -> bool:
    if pd.isna(valeur):
        return False
    return bool(re.fullmatch(r"\d+(\s*\+\s*\d+)+", str(valeur).strip()))


def parse_excel_file(file_path: Path) -> pd.DataFrame:
    # Transforme un fichier Excel annuel en format long.
    year = extract_year_from_filename(file_path)
    xls = pd.ExcelFile(file_path)
    workbook = load_workbook(file_path, data_only=False)

    rows = []

    for sheet_name in xls.sheet_names:
        # Les feuilles Feuil1 / Feuil2 sont génériques ou vides.
        if sheet_name.lower().startswith("feuil"):
            continue

        df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
        worksheet = workbook[sheet_name]

        # Les semaines sont sur la ligne 1, à partir de la colonne 4.
        week_columns = []
        for col in range(4, df.shape[1]):
            week_value = df.iloc[1, col]

            if pd.notna(week_value):
                try:
                    week_number = int(float(week_value))
                    week_columns.append((col, week_number))
                except ValueError:
                    pass

        current_species = None
        current_box = None
        current_temperature = None

        # Les données commencent à la ligne 2.
        for row_idx in range(2, df.shape[0]):
            species = clean_text(df.iloc[row_idx, 0])
            box = clean_text(df.iloc[row_idx, 1])
            temperature = df.iloc[row_idx, 2]
            measure_type = normalize_measure_type(df.iloc[row_idx, 3])

            # Si une nouvelle ligne de boîte/espèce commence, on met à jour le contexte.
            # Si la température est vide au début d'un nouveau bloc, elle reste manquante.
            new_block = species is not None or box is not None

            if species is not None:
                current_species = species

            if box is not None:
                current_box = box

            if new_block:
                current_temperature = temperature if pd.notna(temperature) else None
            elif pd.notna(temperature):
                current_temperature = temperature

            if measure_type is None:
                continue

            for col, week_number in week_columns:
                excel_cell = worksheet.cell(row=row_idx + 1, column=col + 1)
                valeur_brute = df.iloc[row_idx, col]

                rows.append({
                    "annee": year,
                    "groupe": sheet_name,
                    "espece": current_species,
                    "code_boite": current_box,
                    "temperature_cible": current_temperature,
                    "semaine": week_number,
                    "type_mesure": measure_type,
                    "valeur_brute": valeur_brute,
                    "valeur": valeur_brute,
                    "statut_cellule": classify_cell_fill(excel_cell),
                    "couleur_cellule": get_cell_rgb(excel_cell),
                    "fichier_source": file_path.name,
                    "feuille_source": sheet_name,
                    "ligne_source": row_idx + 1,
                    "colonne_source": col + 1,
                })

    return pd.DataFrame(rows)


In [262]:
tracking_long_df = pd.concat(
    [parse_excel_file(file) for file in excel_files],
    ignore_index=True
)

display(tracking_long_df.head(20))
print("Dimensions :", tracking_long_df.shape)
print(tracking_long_df["type_mesure"].value_counts(dropna=False))


,annee,groupe,espece,code_boite,temperature_cible,semaine,type_mesure,valeur_brute,valeur,statut_cellule,couleur_cellule,fichier_source,feuille_source,ligne_source,colonne_source
0,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,1,polypes,NaN,NaN,standard,None,Suivi_2019.xlsx,Aurelia,3,6
1,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,2,polypes,NaN,NaN,standard,None,Suivi_2019.xlsx,Aurelia,3,7
2,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,3,polypes,NaN,NaN,standard,None,Suivi_2019.xlsx,Aurelia,3,8
3,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,4,polypes,NaN,NaN,standard,None,Suivi_2019.xlsx,Aurelia,3,9
4,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,5,polypes,NaN,NaN,standard,None,Suivi_2019.xlsx,Aurelia,3,10
5,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,6,polypes,NaN,NaN,standard,None,Suivi_2019.xlsx,Aurelia,3,11
6,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,7,polypes,NaN,NaN,standard,None,Suivi_2019.xlsx,Aurelia,3,12
7,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,8,polypes,NaN,NaN,standard,None,Suivi_2019.xlsx,Aurelia,3,13
8,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,9,polypes,NaN,NaN,standard,None,Suivi_2019.xlsx,Aurelia,3,14
9,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,10,polypes,NaN,NaN,standard,None,Suivi_2019.xlsx,Aurelia,3,15


Dimensions : (259688, 15)
type_mesure
polypes      130052
ephyrules    129324
nb              312
Name: count, dtype: int64


In [263]:
# Export de contrôle des lignes Staurozoa avant correction.
stauro_values_df = tracking_long_df[
    (tracking_long_df["groupe"] == "Staurozoa")
    & (tracking_long_df["valeur"].notna())
].copy()

stauro_values_df.to_csv(
    ANOMALY_DIR / "staurozoa_a_verifier.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Lignes Staurozoa à vérifier :", stauro_values_df.shape)
display(stauro_values_df.head(20))


Lignes Staurozoa à vérifier : (30, 15)


,annee,groupe,espece,code_boite,temperature_cible,semaine,type_mesure,valeur_brute,valeur,statut_cellule,couleur_cellule,fichier_source,feuille_source,ligne_source,colonne_source
222995,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,20,nb,3.0,3.0,standard,None,Suivi_2025.xlsx,Staurozoa,3,25
222996,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,21,nb,3.0,3.0,standard,None,Suivi_2025.xlsx,Staurozoa,3,26
222997,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,22,nb,3.0,3.0,standard,None,Suivi_2025.xlsx,Staurozoa,3,27
222998,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,23,nb,3,3,standard,None,Suivi_2025.xlsx,Staurozoa,3,28
222999,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,24,nb,3.0,3.0,standard,None,Suivi_2025.xlsx,Staurozoa,3,29
223000,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,25,nb,3.0,3.0,standard,None,Suivi_2025.xlsx,Staurozoa,3,30
223001,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,26,nb,3.0,3.0,standard,None,Suivi_2025.xlsx,Staurozoa,3,31
223002,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,27,nb,3,3,standard,None,Suivi_2025.xlsx,Staurozoa,3,32
223003,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,28,nb,3.0,3.0,standard,None,Suivi_2025.xlsx,Staurozoa,3,33
223004,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,29,nb,3.0,3.0,standard,None,Suivi_2025.xlsx,Staurozoa,3,34


In [264]:
# Correction spécifique pour Staurozoa.

tracking_long_df = tracking_long_df.sort_values(
    ["fichier_source", "feuille_source", "ligne_source", "semaine"]
).copy()

stauro_mask = (
    (tracking_long_df["groupe"] == "Staurozoa")
    & (tracking_long_df["type_mesure"] == "nb")
)

stauro_lignes = (
    tracking_long_df[stauro_mask][["fichier_source", "feuille_source", "ligne_source"]]
    .drop_duplicates()
    .sort_values(["fichier_source", "feuille_source", "ligne_source"])
    .reset_index(drop=True)
)

stauro_lignes["type_corrige"] = np.where(
    stauro_lignes.index % 2 == 0,
    "polypes",
    "ephyrules"
)

tracking_long_df = tracking_long_df.merge(
    stauro_lignes,
    on=["fichier_source", "feuille_source", "ligne_source"],
    how="left"
)

tracking_long_df["type_mesure"] = tracking_long_df["type_corrige"].fillna(
    tracking_long_df["type_mesure"]
)

tracking_long_df = tracking_long_df.drop(columns=["type_corrige"])

print(tracking_long_df["type_mesure"].value_counts(dropna=False))


type_mesure
polypes      130208
ephyrules    129480
Name: count, dtype: int64


In [265]:
clean_tracking_df = tracking_long_df.copy()

# Garder une trace des cellules bleues et grises avant de filtrer les valeurs.
# Bleu : absence de suivi, notamment Covid en 2020.
# Gris : boîte inexistante, pas encore créée ou supprimée.
cellules_bleues_df = clean_tracking_df[
    clean_tracking_df["statut_cellule"].eq("absence_suivi")
].copy()

cellules_grises_df = clean_tracking_df[
    clean_tracking_df["statut_cellule"].eq("boite_inexistante")
].copy()

cellules_bleues_df.to_csv(
    CONTROL_DIR / "cellules_bleues_absence_suivi.csv",
    index=False,
    encoding="utf-8-sig"
)

cellules_grises_df.to_csv(
    CONTROL_DIR / "cellules_grises_boite_inexistante.csv",
    index=False,
    encoding="utf-8-sig"
)

# Garder uniquement les mesures biologiques exploitables.
clean_tracking_df = clean_tracking_df[
    clean_tracking_df["type_mesure"].isin(["polypes", "ephyrules"])
].copy()

# Les cellules grises correspondent à une boîte inexistante ou inactive : on ne les importe pas.
clean_tracking_df = clean_tracking_df[
    ~clean_tracking_df["statut_cellule"].eq("boite_inexistante")
].copy()

# Nettoyer les textes.
for col in ["groupe", "espece", "code_boite", "fichier_source", "feuille_source"]:
    clean_tracking_df[col] = clean_tracking_df[col].astype("string").str.strip()

clean_tracking_df["code_boite"] = (
    clean_tracking_df["code_boite"]
    .str.replace("\n", "", regex=False)
    .str.replace("\r", "", regex=False)
    .str.strip()
)

# Nettoyage des valeurs de relevé.
# Les additions écrites dans Excel sont transformées en total : 5+4 devient 9.
clean_tracking_df["valeur_est_addition"] = clean_tracking_df["valeur_brute"].apply(est_addition_releve)
clean_tracking_df["valeur"] = clean_tracking_df["valeur_brute"].apply(nettoyer_valeur_releve)

valeurs_additionnees_df = clean_tracking_df[
    clean_tracking_df["valeur_est_addition"]
].copy()

valeurs_additionnees_df.to_csv(
    CONTROL_DIR / "valeurs_additionnees.csv",
    index=False,
    encoding="utf-8-sig"
)

clean_tracking_df["temperature_cible"] = pd.to_numeric(
    clean_tracking_df["temperature_cible"],
    errors="coerce"
)

# On ne garde dans les relevés que les cellules qui contiennent une valeur numérique exploitable.
# Les cellules bleues vides restent conservées dans le fichier de contrôle cellules_bleues_absence_suivi.csv.
clean_tracking_df = clean_tracking_df[clean_tracking_df["valeur"].notna()].copy()

print("Données propres :", clean_tracking_df.shape)
print(clean_tracking_df["type_mesure"].value_counts(dropna=False))
print("Espèces manquantes :", clean_tracking_df["espece"].isna().sum())
print("Boîtes manquantes :", clean_tracking_df["code_boite"].isna().sum())
print("Températures manquantes :", clean_tracking_df["temperature_cible"].isna().sum())
print("Valeurs écrites comme additions :", len(valeurs_additionnees_df))
print("Cellules bleues exportées :", len(cellules_bleues_df))
print("Cellules grises exportées :", len(cellules_grises_df))


Données propres : (76242, 16)
type_mesure
polypes      38251
ephyrules    37991
Name: count, dtype: int64
Espèces manquantes : 0
Boîtes manquantes : 0
Températures manquantes : 292
Valeurs écrites comme additions : 58
Cellules bleues exportées : 17
Cellules grises exportées : 11478


In [266]:
def extract_code_espece_local(code_boite):
    if pd.isna(code_boite):
        return None
    parts = str(code_boite).strip().split("-")
    return parts[0] if len(parts) >= 1 else None


def extract_provenance(code_boite):
    if pd.isna(code_boite):
        return None
    parts = str(code_boite).strip().split("-")
    return parts[1] if len(parts) >= 2 else None


def extract_numero_souche_local(code_boite):
    if pd.isna(code_boite):
        return None
    parts = str(code_boite).strip().split("-")
    if len(parts) >= 3:
        return parts[2].split(".")[0]
    return None


def extract_numero_boite_local(code_boite):
    if pd.isna(code_boite):
        return None
    code_boite = str(code_boite).strip()
    return code_boite.split(".")[-1] if "." in code_boite else None


def extract_code_souche(code_boite):
    if pd.isna(code_boite):
        return None
    code_boite = str(code_boite).strip()
    return code_boite.split(".")[0] if "." in code_boite else code_boite


clean_tracking_df["code_espece_local"] = clean_tracking_df["code_boite"].apply(extract_code_espece_local)
clean_tracking_df["code_souche"] = clean_tracking_df["code_boite"].apply(extract_code_souche)
clean_tracking_df["numero_souche_local"] = clean_tracking_df["code_boite"].apply(extract_numero_souche_local)
clean_tracking_df["code_provenance"] = clean_tracking_df["code_boite"].apply(extract_provenance)
clean_tracking_df["numero_boite_local"] = clean_tracking_df["code_boite"].apply(extract_numero_boite_local)

display(clean_tracking_df[[
    "code_boite",
    "code_espece_local",
    "code_souche",
    "numero_souche_local",
    "code_provenance",
    "numero_boite_local",
    "espece"
]].drop_duplicates().head(30))


,code_boite,code_espece_local,code_souche,numero_souche_local,code_provenance,numero_boite_local,espece
21528,ALA-JKA-1.02,ALA,ALA-JKA-1,1,JKA,02,Aurelia labiata
21763,ALA-JKA-1.03,ALA,ALA-JKA-1,1,JKA,03,Aurelia labiata
21988,ALA-JKA-1.04,ALA,ALA-JKA-1,1,JKA,04,Aurelia labiata
22256,ALI-JKA-1.03,ALI,ALI-JKA-1,1,JKA,03,Aurelia limbata
22408,ALI-JKA-1.04,ALI,ALI-JKA-1,1,JKA,04,Aurelia limbata
22590,ALI-JKA-2.02,ALI,ALI-JKA-2,2,JKA,02,Aurelia limbata
22776,ALI-FPA-3.01,ALI,ALI-FPA-3,3,FPA,01,Aurelia limbata
23088,AMA-JKA-1.01,AMA,AMA-JKA-1,1,JKA,01,Aurelia maldivensis
23192,AMA-JKA-1.03,AMA,AMA-JKA-1,1,JKA,03,Aurelia maldivensis
23409,AMA-JKA-1.04,AMA,AMA-JKA-1,1,JKA,04,Aurelia maldivensis


In [267]:
species_corrections = {
    "Alatina morandini": "Alatina morandinii",
    "Tiarropsis multicirrata": "Tiaropsis multicirrata",
    "Chrysaoa achlyos": "Chrysaora achlyos",
    "Cyanea lamarcki": "Cyanea lamarckii",
    "Clytia hemispherica": "Clytia hemisphaerica",
    "Cyanea nozakii Kishinouye": "Cyanea nozakii",
    "eudendryum sp": "Eudendrium sp.",
}

# Garder une trace des corrections effectuées.
species_corrections_df = pd.DataFrame([
    {"nom_initial": initial, "nom_corrige": corrected}
    for initial, corrected in species_corrections.items()
])

species_corrections_df.to_csv(
    ANOMALY_DIR / "corrections_noms_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

clean_tracking_df["espece"] = clean_tracking_df["espece"].replace(species_corrections)

display(species_corrections_df)


,nom_initial,nom_corrige
0,Alatina morandini,Alatina morandinii
1,Tiarropsis multicirrata,Tiaropsis multicirrata
2,Chrysaoa achlyos,Chrysaora achlyos
3,Cyanea lamarcki,Cyanea lamarckii
4,Clytia hemispherica,Clytia hemisphaerica
5,Cyanea nozakii Kishinouye,Cyanea nozakii
6,eudendryum sp,Eudendrium sp.


In [268]:
# Anomalie 1 : températures manquantes.
anomalies_temperature_df = clean_tracking_df[
    clean_tracking_df["temperature_cible"].isna()
].copy()

anomalies_temperature_df.to_csv(
    ANOMALY_DIR / "anomalies_temperature_manquante.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Lignes avec température manquante :", len(anomalies_temperature_df))

display(
    anomalies_temperature_df[[
        "fichier_source",
        "feuille_source",
        "code_boite",
        "espece",
        "annee",
        "semaine"
    ]]
    .drop_duplicates()
    .sort_values(["fichier_source", "feuille_source", "code_boite", "semaine"])
    .head(30)
)


Lignes avec température manquante : 292


,fichier_source,feuille_source,code_boite,espece,annee,semaine
183664,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,1
183665,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,2
183666,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,3
183667,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,4
183668,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,5
183669,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,6
183670,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,7
183671,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,8
183672,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,9
183673,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,10


In [269]:
# Anomalie 2 : même code de souche associé à plusieurs espèces.
species_per_souche = (
    clean_tracking_df[["code_souche", "espece"]]
    .drop_duplicates()
    .groupby("code_souche")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
    .sort_values("nb_especes", ascending=False)
)

problem_souches = species_per_souche[
    species_per_souche["nb_especes"] > 1
]["code_souche"].tolist()

anomalies_souche_df = (
    clean_tracking_df[
        clean_tracking_df["code_souche"].isin(problem_souches)
    ][[
        "code_souche",
        "code_boite",
        "espece",
        "fichier_source",
        "feuille_source"
    ]]
    .drop_duplicates()
    .sort_values(["code_souche", "espece", "code_boite"])
)

anomalies_souche_df.to_csv(
    ANOMALY_DIR / "anomalies_souche_plusieurs_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Codes de souche associés à plusieurs espèces :", len(problem_souches))
display(species_per_souche[species_per_souche["nb_especes"] > 1])
display(anomalies_souche_df.head(50))


Codes de souche associés à plusieurs espèces : 11


,code_souche,nb_especes
32,CAC-JKA-2,2
27,AVA-TAI-1,2
197,THY-FGU-1,2
47,CFU-JKA-2,2
130,LRO-MAL-1,2
25,ATH-THA-1,2
147,OCE-TAI-1,2
183,SME-JKA-1,2
181,SMA-MAU-3,2
180,SMA-JKA-2,2


,code_souche,code_boite,espece,fichier_source,feuille_source
187453,ATH-THA-1,ATH-THA-1.01,Aurelia sp. THAI,Suivi_2025.xlsx,Aurelia
227552,ATH-THA-1,ATH-THA-1.01,Aurelia sp.4,Suivi_2026.xlsx,Aurelia
226512,AVA-TAI-1,AVA-TAI-1.01,Aurelia malayensis,Suivi_2026.xlsx,Aurelia
226824,AVA-TAI-1,AVA-TAI-1.02,Aurelia malayensis,Suivi_2026.xlsx,Aurelia
149701,AVA-TAI-1,AVA-TAI-1.01,"Aurelia sp.2 ""Valentine""",Suivi_2024.xlsx,Aurelia
186500,AVA-TAI-1,AVA-TAI-1.01,"Aurelia sp.2 ""Valentine""",Suivi_2025.xlsx,Aurelia
186689,AVA-TAI-1,AVA-TAI-1.02,"Aurelia sp.2 ""Valentine""",Suivi_2025.xlsx,Aurelia
99370,CAC-JKA-2,CAC-JKA-2.01,Chrysaora achlyos,Suivi_2022.xlsx,Semaestomeae
134368,CAC-JKA-2,CAC-JKA-2.01,Chrysaora achlyos,Suivi_2023.xlsx,Semaestomeae
172432,CAC-JKA-2,CAC-JKA-2.01,Chrysaora achlyos,Suivi_2024.xlsx,Semaestomeae


In [270]:
# Anomalie 3 : même code de boîte associé à plusieurs espèces.
species_per_boite = (
    clean_tracking_df[["code_boite", "espece"]]
    .drop_duplicates()
    .groupby("code_boite")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
    .sort_values("nb_especes", ascending=False)
)

codes_boites_ambigues = species_per_boite[
    species_per_boite["nb_especes"] > 1
]["code_boite"].tolist()

anomalies_boites_df = (
    clean_tracking_df[
        clean_tracking_df["code_boite"].isin(codes_boites_ambigues)
    ][[
        "code_boite",
        "code_souche",
        "espece",
        "fichier_source",
        "feuille_source"
    ]]
    .drop_duplicates()
    .sort_values(["code_boite", "espece", "fichier_source"])
)

anomalies_boites_df.to_csv(
    ANOMALY_DIR / "anomalies_boites_plusieurs_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Codes de boîte associés à plusieurs espèces :", len(codes_boites_ambigues))
display(species_per_boite[species_per_boite["nb_especes"] > 1])
display(anomalies_boites_df.head(50))


Codes de boîte associés à plusieurs espèces : 14


,code_boite,nb_especes
503,SME-JKA-1.01,2
123,CCP-TPR-1.08,2
497,SMA-JKA-2.06,2
496,SMA-JKA-2.05,2
504,SME-JKA-1.04,2
505,SME-JKA-1.05,2
72,AVA-TAI-1.02,2
71,AVA-TAI-1.01,2
68,ATH-THA-1.01,2
345,LRO-MAL-1.01,2


,code_boite,code_souche,espece,fichier_source,feuille_source
187453,ATH-THA-1.01,ATH-THA-1,Aurelia sp. THAI,Suivi_2025.xlsx,Aurelia
227552,ATH-THA-1.01,ATH-THA-1,Aurelia sp.4,Suivi_2026.xlsx,Aurelia
226512,AVA-TAI-1.01,AVA-TAI-1,Aurelia malayensis,Suivi_2026.xlsx,Aurelia
149701,AVA-TAI-1.01,AVA-TAI-1,"Aurelia sp.2 ""Valentine""",Suivi_2024.xlsx,Aurelia
186500,AVA-TAI-1.01,AVA-TAI-1,"Aurelia sp.2 ""Valentine""",Suivi_2025.xlsx,Aurelia
226824,AVA-TAI-1.02,AVA-TAI-1,Aurelia malayensis,Suivi_2026.xlsx,Aurelia
186689,AVA-TAI-1.02,AVA-TAI-1,"Aurelia sp.2 ""Valentine""",Suivi_2025.xlsx,Aurelia
173268,CCP-TPR-1.08,CCP-TPR-1,Chrysaora chesapeakei,Suivi_2024.xlsx,Semaestomeae
214552,CCP-TPR-1.08,CCP-TPR-1,"Chrysaora chesapeakei ""white""",Suivi_2025.xlsx,Semaestomeae
245336,LRO-MAL-1.01,LRO-MAL-1,Lobonemoides gracilis,Suivi_2026.xlsx,Rhizostomae


In [271]:
# ============================================================
# Séparation automatique : anomalies bloquantes / validation métier
# ============================================================

# On part de toutes les données nettoyées.
df = clean_tracking_df.copy()

# 1. Codes de boîte associés à plusieurs noms d'espèces
boites_plusieurs_especes = (
    df[["code_boite", "espece"]]
    .dropna()
    .drop_duplicates()
    .groupby("code_boite")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
)

codes_boites_ambigues = boites_plusieurs_especes[
    boites_plusieurs_especes["nb_especes"] > 1
]["code_boite"].tolist()

# 2. Codes de souche associés à plusieurs noms d'espèces
souches_plusieurs_especes = (
    df[["code_souche", "espece"]]
    .dropna()
    .drop_duplicates()
    .groupby("code_souche")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
)

codes_souches_ambigues = souches_plusieurs_especes[
    souches_plusieurs_especes["nb_especes"] > 1
]["code_souche"].tolist()

# 3. Codes espèce locaux associés à plusieurs noms scientifiques.
# Après la réunion métier, ce cas n'est plus forcément une erreur :
# le nom scientifique peut évoluer, tandis que le code espèce sert d'identifiant stable.
codes_especes_plusieurs_noms = (
    df[["code_espece_local", "espece"]]
    .dropna()
    .drop_duplicates()
    .groupby("code_espece_local")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
)

codes_especes_ambigus = codes_especes_plusieurs_noms[
    codes_especes_plusieurs_noms["nb_especes"] > 1
]["code_espece_local"].tolist()

# 4. Codes de boîte non standards.
# Un code standard ressemble à : ABC-XXX-1.01
masque_code_boite_non_standard = ~df["code_boite"].astype(str).str.match(
    r"^[A-Z]{3}-[A-Z0-9?]{3}-[0-9]+(\.[0-9]+)?$",
    na=False
)

# 5. Températures manquantes.
# Elles restent bloquantes pour le rangement en zone thermique tant qu'elles ne sont pas validées.
masque_temperature_manquante = df["temperature_cible"].isna()

# 6. Noms scientifiques associés à plusieurs codes espèce.
# Exemple : Obelia sp. avec les codes OBE et OSP.
noms_plusieurs_codes = (
    df[["espece", "code_espece_local"]]
    .dropna()
    .drop_duplicates()
    .groupby("espece")["code_espece_local"]
    .nunique()
    .reset_index(name="nb_codes")
)

noms_especes_ambigus = noms_plusieurs_codes[
    noms_plusieurs_codes["nb_codes"] > 1
]["espece"].tolist()

# On distingue les anomalies vraiment bloquantes des cas à valider.
df["raison_anomalie_bloquante"] = ""
df["raison_validation_metier"] = ""

df.loc[masque_code_boite_non_standard, "raison_anomalie_bloquante"] += "code_boite_non_standard;"
df.loc[masque_temperature_manquante, "raison_anomalie_bloquante"] += "temperature_manquante;"

# Ces cas sont exportés pour validation, mais ne sont plus exclus automatiquement.
df.loc[df["code_boite"].isin(codes_boites_ambigues), "raison_validation_metier"] += "boite_plusieurs_noms_especes;"
df.loc[df["code_souche"].isin(codes_souches_ambigues), "raison_validation_metier"] += "souche_plusieurs_noms_especes;"
df.loc[df["code_espece_local"].isin(codes_especes_ambigus), "raison_validation_metier"] += "code_espece_plusieurs_noms;"
df.loc[df["espece"].isin(noms_especes_ambigus), "raison_validation_metier"] += "nom_scientifique_plusieurs_codes;"

masque_anomalie_bloquante = df["raison_anomalie_bloquante"].ne("")
masque_validation_metier = df["raison_validation_metier"].ne("")

# Données valides : on retire seulement les anomalies bloquantes.
# Les évolutions de noms d'espèces restent disponibles pour construire les tables,
# en gardant un code espèce stable et le dernier nom observé.
clean_tracking_valid_df = df[~masque_anomalie_bloquante].copy()
clean_tracking_anomalies_df = df[masque_anomalie_bloquante].copy()
validation_metier_df = df[masque_validation_metier].copy()

print("Lignes totales :", len(df))
print("Lignes valides :", len(clean_tracking_valid_df))
print("Anomalies bloquantes :", len(clean_tracking_anomalies_df))
print("Lignes à validation métier :", len(validation_metier_df))

print("Boîtes avec plusieurs noms d'espèces :", len(codes_boites_ambigues))
print("Souches avec plusieurs noms d'espèces :", len(codes_souches_ambigues))
print("Codes espèce avec plusieurs noms :", len(codes_especes_ambigus))
print("Codes boîte non standards :", int(masque_code_boite_non_standard.sum()))
print("Températures manquantes :", int(masque_temperature_manquante.sum()))
print("Noms scientifiques avec plusieurs codes :", len(noms_especes_ambigus))

display(noms_plusieurs_codes[noms_plusieurs_codes["nb_codes"] > 1])


Lignes totales : 76242
Lignes valides : 74332
Anomalies bloquantes : 1910
Lignes à validation métier : 18343
Boîtes avec plusieurs noms d'espèces : 14
Souches avec plusieurs noms d'espèces : 11
Codes espèce avec plusieurs noms : 16
Codes boîte non standards : 1618
Températures manquantes : 292
Noms scientifiques avec plusieurs codes : 10


,espece,nb_codes
21,Aurelia sp.4,2
39,"Chrysaora chesapeakei ""white""",2
57,Cotylorhiza tuberculata,2
79,Hydrozoa sp.,4
85,Linuche aquila,3
95,Melicertum octocostatum,2
100,Obelia sp.,2
104,Pennaria disticha,2
110,Rhizostoma luteum,2
124,Stomolophus sp.2,2


In [272]:
# ============================================================
# Export des anomalies et des validations métier
# ============================================================

ANOMALY_DIR = OUTPUT_DIR / "anomalies"
ANOMALY_DIR.mkdir(parents=True, exist_ok=True)

colonnes_decision = {
    "decision_aquarium": "",
    "correction_code_espece": "",
    "correction_nom_scientifique": "",
    "correction_code_souche": "",
    "correction_code_boite": "",
    "temperature_validee": "",
    "commentaire_aquarium": "",
}

# Anomalies bloquantes : elles sont exclues des CSV finaux.
anomalies_bloquantes_export_df = clean_tracking_anomalies_df.copy()
for colonne, valeur in colonnes_decision.items():
    anomalies_bloquantes_export_df[colonne] = valeur

anomalies_bloquantes_export_df.to_csv(
    ANOMALY_DIR / "anomalies_bloquantes_toutes.csv",
    index=False,
    encoding="utf-8-sig"
)

# Fichier historique gardé pour compatibilité avec les anciennes versions du notebook.
anomalies_bloquantes_export_df.to_csv(
    ANOMALY_DIR / "anomalies_toutes.csv",
    index=False,
    encoding="utf-8-sig"
)

# Validations métier : ces lignes sont conservées pour les tests,
# mais elles doivent être vérifiées par l'aquarium.
validation_metier_export_df = validation_metier_df.copy()
for colonne, valeur in colonnes_decision.items():
    validation_metier_export_df[colonne] = valeur

validation_metier_export_df.to_csv(
    ANOMALY_DIR / "validations_metier_toutes.csv",
    index=False,
    encoding="utf-8-sig"
)

# Version Excel pratique pour que l'aquarium puisse répondre directement dessus.
validation_metier_export_df.to_excel(
    ANOMALY_DIR / "validations_metier_a_completer.xlsx",
    index=False
)

# Fichiers par type, pour faciliter la lecture.
df[df["code_boite"].isin(codes_boites_ambigues)].to_csv(
    ANOMALY_DIR / "validations_boites_plusieurs_noms_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

df[df["code_souche"].isin(codes_souches_ambigues)].to_csv(
    ANOMALY_DIR / "validations_souches_plusieurs_noms_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

df[df["code_espece_local"].isin(codes_especes_ambigus)].to_csv(
    ANOMALY_DIR / "validations_codes_especes_plusieurs_noms.csv",
    index=False,
    encoding="utf-8-sig"
)

df[df["espece"].isin(noms_especes_ambigus)].to_csv(
    ANOMALY_DIR / "validations_noms_especes_plusieurs_codes.csv",
    index=False,
    encoding="utf-8-sig"
)

# Anomalies bloquantes par type.
anomalies_codes_non_standards_df = df[masque_code_boite_non_standard].copy()
anomalies_codes_non_standards_df.to_csv(
    ANOMALY_DIR / "anomalies_codes_boite_non_standards.csv",
    index=False,
    encoding="utf-8-sig"
)

anomalies_temperatures_df = df[masque_temperature_manquante].copy()
anomalies_temperatures_df.to_csv(
    ANOMALY_DIR / "anomalies_temperatures_manquantes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fichiers anomalies exportés dans :", ANOMALY_DIR)
print("Anomalies bloquantes :", len(clean_tracking_anomalies_df))
print("Validations métier :", len(validation_metier_df))


Fichiers anomalies exportés dans : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase/anomalies
Anomalies bloquantes : 1910
Validations métier : 18343


In [273]:
# Table ESPECE, à partir des données valides.
# Le code espèce est utilisé comme identifiant stable.
# Si un code espèce a eu plusieurs noms dans le temps, on retient le dernier nom observé.

historique_noms_especes_df = (
    clean_tracking_valid_df[[
        "code_espece_local",
        "espece",
        "annee",
        "semaine",
        "fichier_source",
        "feuille_source"
    ]]
    .dropna(subset=["code_espece_local", "espece"])
    .drop_duplicates()
    .sort_values(["code_espece_local", "annee", "semaine"])
)

historique_noms_especes_df.to_csv(
    CONTROL_DIR / "historique_noms_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

espece_df = (
    historique_noms_especes_df
    .sort_values(["code_espece_local", "annee", "semaine"])
    .drop_duplicates(subset=["code_espece_local"], keep="last")
    [["code_espece_local", "espece"]]
    .sort_values("code_espece_local")
    .reset_index(drop=True)
)

espece_df["id_espece"] = range(1, len(espece_df) + 1)

espece_df = espece_df.rename(columns={
    "espece": "nom_scientifique",
    "code_espece_local": "code_espece"
})

espece_df = espece_df[[
    "id_espece",
    "code_espece",
    "nom_scientifique"
]]

print("Nombre d'espèces :", len(espece_df))
print("Doublons code espèce :", espece_df["code_espece"].duplicated().sum())
print("Doublons nom scientifique :", espece_df["nom_scientifique"].duplicated().sum())
print("Historique des noms exporté dans :", CONTROL_DIR / "historique_noms_especes.csv")
display(espece_df.head(20))


Nombre d'espèces : 126
Doublons code espèce : 0
Doublons nom scientifique : 3
Historique des noms exporté dans : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase/controles/historique_noms_especes.csv


,id_espece,code_espece,nom_scientifique
0,1,AAL,Alatina alata
1,2,AAU,Aurelia aurita
2,3,ACA,Aurelia coerulea
3,4,ACO,Aequorea coerulescens
4,5,ADD,Aurelia coerulea DD
5,6,ADI,Aglaophenia diegensis
6,7,AFL,Acromitus flagellatus
7,8,AHA,Acromitus hardenbergi
8,9,ALA,Aurelia labiata
9,10,ALI,Aurelia limbata


In [274]:
# Table SOUCHE.
# Une souche est identifiée par son code_souche.
# Le lien vers l'espèce se fait par le code_espece_local extrait du code de boîte,
# pas uniquement par le nom scientifique, car ce nom peut évoluer.

souche_source_df = (
    clean_tracking_valid_df[[
        "code_souche",
        "numero_souche_local",
        "code_provenance",
        "code_espece_local",
        "annee",
        "semaine"
    ]]
    .dropna(subset=["code_souche", "code_espece_local"])
    .drop_duplicates()
    .sort_values(["code_souche", "annee", "semaine"])
)

souche_df = (
    souche_source_df
    .drop_duplicates(subset=["code_souche"], keep="last")
    .reset_index(drop=True)
)

souche_df = souche_df.merge(
    espece_df[["id_espece", "code_espece"]],
    left_on="code_espece_local",
    right_on="code_espece",
    how="left"
)

souche_df["id_souche"] = range(1, len(souche_df) + 1)

souche_df = souche_df[[
    "id_souche",
    "code_souche",
    "numero_souche_local",
    "code_provenance",
    "id_espece"
]]

display(souche_df.head(20))
print("Nombre de souches :", len(souche_df))
print("Doublons code_souche :", souche_df["code_souche"].duplicated().sum())
print("Souches sans espèce :", souche_df["id_espece"].isna().sum())


,id_souche,code_souche,numero_souche_local,code_provenance,id_espece
0,1,AAL-FGU-1,1,FGU,1
1,2,AAU-NBE-1,1,NBE,2
2,3,ACA-FBA-1,1,FBA,3
3,4,ACA-FBA-2,2,FBA,3
4,5,ACO-JKA-2,2,JKA,4
5,6,ADD-JTB-1,1,JTB,5
6,7,ADI-MGF-1,1,MGF,6
7,8,AFL-TAI-1,1,TAI,7
8,9,AHA-MAL-1,1,MAL,8
9,10,ALA-JKA-1,1,JKA,9


Nombre de souches : 194
Doublons code_souche : 0
Souches sans espèce : 0


In [275]:
# Table BOITE.
# Une boîte est identifiée par son code local.
# Le lien vers la souche se fait par code_souche, afin de rester stable même si le nom d'espèce évolue.

boite_source_df = (
    clean_tracking_valid_df[[
        "code_boite",
        "code_souche",
        "numero_boite_local",
        "annee",
        "semaine"
    ]]
    .dropna(subset=["code_boite", "code_souche"])
    .drop_duplicates()
    .sort_values(["code_boite", "annee", "semaine"])
)

boite_df = (
    boite_source_df
    .drop_duplicates(subset=["code_boite"], keep="last")
    .reset_index(drop=True)
)

boite_df = boite_df.merge(
    souche_df[["id_souche", "code_souche"]],
    on="code_souche",
    how="left"
)

boite_df["id_boite"] = range(1, len(boite_df) + 1)
boite_df["code_local"] = boite_df["code_boite"]

boite_df = boite_df[[
    "id_boite",
    "code_local",
    "numero_boite_local",
    "id_souche"
]]

display(boite_df.head(20))
print("Nombre de boîtes :", len(boite_df))
print("Doublons code_local :", boite_df["code_local"].duplicated().sum())
print("Boîtes sans souche :", boite_df["id_souche"].isna().sum())


,id_boite,code_local,numero_boite_local,id_souche
0,1,AAL-FGU-1.01,01,1
1,2,AAL-FGU-1.02,02,1
2,3,AAU-NBE-1.01,01,2
3,4,AAU-NBE-1.02,02,2
4,5,ACA-FBA-1.01,01,3
5,6,ACA-FBA-1.02,02,3
6,7,ACA-FBA-2.01,01,4
7,8,ACA-FBA-2.02,02,4
8,9,ACO-JKA-2.01,01,5
9,10,ACO-JKA-2.02,02,5


Nombre de boîtes : 555
Doublons code_local : 0
Boîtes sans souche : 0


In [276]:
# Table SAISIR_RELEVE, à partir des données valides uniquement.

releve_base_df = clean_tracking_valid_df.merge(
    boite_df[["id_boite", "code_local"]],
    left_on="code_boite",
    right_on="code_local",
    how="left"
)

print("Lignes de relevés sans boîte :", releve_base_df["id_boite"].isna().sum())

releve_df = (
    releve_base_df
    .pivot_table(
        index=["annee", "semaine", "id_boite"],
        columns="type_mesure",
        values="valeur",
        aggfunc="first"
    )
    .reset_index()
)

releve_df = releve_df.rename(columns={
    "polypes": "nombre_polypes",
    "ephyrules": "nombre_ephyrules"
})

releve_df["date_releve"] = (
    releve_df["annee"].astype(int).astype(str)
    + "-S"
    + releve_df["semaine"].astype(int).astype(str).str.zfill(2)
)

releve_df = releve_df[[
    "id_boite",
    "annee",
    "semaine",
    "date_releve",
    "nombre_polypes",
    "nombre_ephyrules"
]]

display(releve_df.head(20))
print("Nombre de relevés :", len(releve_df))
print("Relevés sans boîte :", releve_df["id_boite"].isna().sum())
print("Relevés sans polypes :", releve_df["nombre_polypes"].isna().sum())
print("Relevés sans éphyrules :", releve_df["nombre_ephyrules"].isna().sum())


Lignes de relevés sans boîte : 0


type_mesure,id_boite,annee,semaine,date_releve,nombre_polypes,nombre_ephyrules
0,9,2020,1,2020-S01,100,0
1,10,2020,1,2020-S01,100,0
2,21,2020,1,2020-S01,200,200
3,22,2020,1,2020-S01,50,0
4,30,2020,1,2020-S01,200,200
5,33,2020,1,2020-S01,200,0
6,35,2020,1,2020-S01,100,0
7,39,2020,1,2020-S01,200,0
8,40,2020,1,2020-S01,200,0
9,51,2020,1,2020-S01,200,3


Nombre de relevés : 37102
Relevés sans boîte : 0
Relevés sans polypes : 56
Relevés sans éphyrules : 208


In [277]:
# Contrôle : relevés incomplets.
# Ce ne sont pas forcément des anomalies bloquantes :
# une seule des deux mesures peut être absente dans les fichiers historiques.
anomalies_releves_incomplets_df = releve_df[
    releve_df["nombre_polypes"].isna() | releve_df["nombre_ephyrules"].isna()
].copy()

anomalies_releves_incomplets_df.to_csv(
    ANOMALY_DIR / "anomalies_releves_incomplets.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Relevés incomplets :", len(anomalies_releves_incomplets_df))
print("Ces lignes sont conservées si au moins une mesure biologique est présente.")
display(anomalies_releves_incomplets_df.head(20))


Relevés incomplets : 264
Ces lignes sont conservées si au moins une mesure biologique est présente.


type_mesure,id_boite,annee,semaine,date_releve,nombre_polypes,nombre_ephyrules
1206,315,2020,15,2020-S15,NaN,0
8415,388,2021,41,2021-S41,NaN,0
8516,388,2021,42,2021-S42,NaN,0
8617,388,2021,43,2021-S43,NaN,0
8701,252,2021,44,2021-S44,NaN,50
8718,388,2021,44,2021-S44,NaN,0
8743,546,2021,44,2021-S44,NaN,0
8819,388,2021,45,2021-S45,NaN,0
8844,546,2021,45,2021-S45,NaN,0
8920,388,2021,46,2021-S46,NaN,0


In [278]:
# Table ZONE_THERMIQUE, à partir des données valides uniquement.

zone_thermique_df = (
    clean_tracking_valid_df[["temperature_cible"]]
    .dropna()
    .drop_duplicates()
    .sort_values("temperature_cible")
    .reset_index(drop=True)
)

zone_thermique_df["id_zone"] = range(1, len(zone_thermique_df) + 1)
zone_thermique_df["nom_zone"] = (
    "Zone "
    + zone_thermique_df["temperature_cible"].astype(str)
    + "°C"
)

zone_thermique_df = zone_thermique_df[[
    "id_zone",
    "nom_zone",
    "temperature_cible"
]]

display(zone_thermique_df)
print("Nombre de zones thermiques :", len(zone_thermique_df))


,id_zone,nom_zone,temperature_cible
0,1,Zone 5.0°C,5.0
1,2,Zone 10.0°C,10.0
2,3,Zone 15.0°C,15.0
3,4,Zone 20.0°C,20.0
4,5,Zone 25.0°C,25.0
5,6,Zone 30.0°C,30.0


Nombre de zones thermiques : 6


In [279]:
# Les températures manquantes sont actuellement exclues des données valides.
# Le fichier anomalies_temperatures_manquantes.csv permet à l'aquarium de décider
# si certaines lignes doivent être corrigées puis réintégrées.
print("Températures manquantes exclues :", len(anomalies_temperatures_df))


Températures manquantes exclues : 292


In [280]:
# Table RANGE, à partir des données valides avec température renseignée.

range_base_df = clean_tracking_valid_df[
    clean_tracking_valid_df["temperature_cible"].notna()
].copy()

range_base_df = range_base_df.merge(
    boite_df[["id_boite", "code_local"]],
    left_on="code_boite",
    right_on="code_local",
    how="left"
)

range_base_df = range_base_df.merge(
    zone_thermique_df[["id_zone", "temperature_cible"]],
    on="temperature_cible",
    how="left"
)

range_df = (
    range_base_df[[
        "id_boite",
        "id_zone",
        "annee",
        "semaine"
    ]]
    .dropna(subset=["id_boite", "id_zone"])
    .drop_duplicates()
    .reset_index(drop=True)
)

range_df["date_entree"] = (
    range_df["annee"].astype(int).astype(str)
    + "-S"
    + range_df["semaine"].astype(int).astype(str).str.zfill(2)
)

range_df = range_df[[
    "id_boite",
    "id_zone",
    "annee",
    "semaine",
    "date_entree"
]]

display(range_df.head(20))
print("Nombre de rangements :", len(range_df))
print("Rangements sans boîte :", range_df["id_boite"].isna().sum())
print("Rangements sans zone :", range_df["id_zone"].isna().sum())


,id_boite,id_zone,annee,semaine,date_entree
0,21,1,2020,1,2020-S01
1,21,1,2020,2,2020-S02
2,21,1,2020,3,2020-S03
3,21,1,2020,4,2020-S04
4,21,1,2020,5,2020-S05
5,21,1,2020,6,2020-S06
6,21,1,2020,7,2020-S07
7,21,1,2020,8,2020-S08
8,21,1,2020,9,2020-S09
9,21,1,2020,10,2020-S10


Nombre de rangements : 37131
Rangements sans boîte : 0
Rangements sans zone : 0


In [281]:
espece_df.to_csv(OUTPUT_DIR / "espece.csv", index=False, encoding="utf-8-sig")
souche_df.to_csv(OUTPUT_DIR / "souche.csv", index=False, encoding="utf-8-sig")
boite_df.to_csv(OUTPUT_DIR / "boite.csv", index=False, encoding="utf-8-sig")
releve_df.to_csv(OUTPUT_DIR / "saisir_releve.csv", index=False, encoding="utf-8-sig")
zone_thermique_df.to_csv(OUTPUT_DIR / "zone_thermique.csv", index=False, encoding="utf-8-sig")
range_df.to_csv(OUTPUT_DIR / "range.csv", index=False, encoding="utf-8-sig")

print("Exports terminés dans :", OUTPUT_DIR)
print("Anomalies exportées dans :", ANOMALY_DIR)
print("Contrôles exportés dans :", CONTROL_DIR)


Exports terminés dans : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase
Anomalies exportées dans : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase/anomalies
Contrôles exportés dans : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase/controles


In [282]:
# ============================================================
# Vérification finale avant import Django
# ============================================================

resume_controles = {
    "nb_especes": len(espece_df),
    "nb_souches": len(souche_df),
    "nb_boites_valides": len(boite_df),
    "nb_releves": len(releve_df),
    "nb_zones_thermiques": len(zone_thermique_df),
    "nb_rangements": len(range_df),
    "doublons_code_espece": int(espece_df["code_espece"].duplicated().sum()),
    "doublons_code_souche": int(souche_df["code_souche"].duplicated().sum()),
    "doublons_code_local": int(boite_df["code_local"].duplicated().sum()),
    "souches_sans_espece": int(souche_df["id_espece"].isna().sum()),
    "boites_sans_souche": int(boite_df["id_souche"].isna().sum()),
    "releves_sans_boite": int(releve_base_df["id_boite"].isna().sum()),
    "rangements_sans_boite": int(range_df["id_boite"].isna().sum()),
    "rangements_sans_zone": int(range_df["id_zone"].isna().sum()),
    "anomalies_bloquantes_total": len(clean_tracking_anomalies_df),
    "validations_metier_total": len(validation_metier_df),
    "anomalies_temperature": len(anomalies_temperatures_df),
    "anomalies_codes_boite_non_standards": len(anomalies_codes_non_standards_df),
    "validations_boites_plusieurs_noms_especes": len(codes_boites_ambigues),
    "validations_souches_plusieurs_noms_especes": len(codes_souches_ambigues),
    "validations_codes_especes_plusieurs_noms": len(codes_especes_ambigus),
    "validations_noms_especes_plusieurs_codes": len(noms_especes_ambigus),
    "controles_releves_incomplets": len(anomalies_releves_incomplets_df),
    "controles_valeurs_additionnees": len(valeurs_additionnees_df),
    "controles_cellules_bleues_absence_suivi": len(cellules_bleues_df),
    "controles_cellules_grises_boite_inexistante": len(cellules_grises_df),
}

resume_controles_df = pd.DataFrame(
    resume_controles.items(),
    columns=["controle", "valeur"]
)

resume_controles_df.to_csv(
    CONTROL_DIR / "resume_controles.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resume_controles_df)


,controle,valeur
0,nb_especes,126
1,nb_souches,194
2,nb_boites_valides,555
3,nb_releves,37102
4,nb_zones_thermiques,6
5,nb_rangements,37131
6,doublons_code_espece,0
7,doublons_code_souche,0
8,doublons_code_local,0
9,souches_sans_espece,0


In [283]:
# Les CSV finaux sont dans OUTPUT_DIR.
# Les anomalies bloquantes sont dans ANOMALY_DIR.
# Les fichiers de contrôle sont dans CONTROL_DIR.
print("CSV finaux :", OUTPUT_DIR)
print("Fichiers anomalies :", ANOMALY_DIR)
print("Fichiers de contrôle :", CONTROL_DIR)


CSV finaux : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase
Fichiers anomalies : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase/anomalies
Fichiers de contrôle : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase/controles


In [284]:
# Vérification rapide des fichiers exportés.
for csv_file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(csv_file.name)

print("--- anomalies ---")
for file in sorted(ANOMALY_DIR.glob("*")):
    print(file.name)

print("--- controles ---")
for file in sorted(CONTROL_DIR.glob("*")):
    print(file.name)


boite.csv
espece.csv
range.csv
saisir_releve.csv
souche.csv
zone_thermique.csv
--- anomalies ---
.DS_Store
anomalies_bloquantes_toutes.csv
anomalies_boites_ambigues.csv
anomalies_boites_plusieurs_especes.csv
anomalies_codes_boite_non_standards.csv
anomalies_codes_especes_ambigus.csv
anomalies_noms_especes_ambigus.csv
anomalies_releves_incomplets.csv
anomalies_souche_plusieurs_especes.csv
anomalies_souches_ambigues.csv
anomalies_temperature_manquante.csv
anomalies_temperatures_manquantes.csv
anomalies_toutes.csv
corrections_noms_especes.csv
staurozoa_a_verifier.csv
validations_boites_plusieurs_noms_especes.csv
validations_codes_especes_plusieurs_noms.csv
validations_metier_a_completer.xlsx
validations_metier_toutes.csv
validations_noms_especes_plusieurs_codes.csv
validations_souches_plusieurs_noms_especes.csv
--- controles ---
cellules_bleues_absence_suivi.csv
cellules_grises_boite_inexistante.csv
historique_noms_especes.csv
resume_controles.csv
valeurs_additionnees.csv


In [287]:
# ============================================================
# Affichage dans le notebook :
# - valeurs écrites sous forme d'addition
# - températures manquantes
# ============================================================

from pathlib import Path
import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from IPython.display import display, Markdown

DATA_DIR = Path("/Users/akkouh/Desktop/polybase")

EXCEL_FILES = [
    DATA_DIR / "Suivi_2020.xlsx",
    DATA_DIR / "Suivi_2021.xlsx",
    DATA_DIR / "Suivi_2022.xlsx",
    DATA_DIR / "Suivi_2023.xlsx",
    DATA_DIR / "Suivi_2024.xlsx",
    DATA_DIR / "Suivi_2025.xlsx",
    DATA_DIR / "Suivi_2026.xlsx",
]

REGEX_ADDITION = re.compile(r"^\s*\d+(?:\s*\+\s*\d+)+\s*$")


def valeur_est_vide(valeur):
    if valeur is None:
        return True
    if isinstance(valeur, str) and valeur.strip() == "":
        return True
    return False


def calculer_addition(valeur):
    return sum(int(x.strip()) for x in str(valeur).replace(" ", "").split("+"))


valeurs_additionnees = []
temperatures_manquantes = []

for excel_path in EXCEL_FILES:
    if not excel_path.exists():
        print(f"Fichier introuvable : {excel_path}")
        continue

    wb = load_workbook(excel_path, data_only=False)

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]

        if ws.max_row < 3 or ws.max_column < 5:
            continue

        for row in range(3, ws.max_row + 1):
            espece = ws.cell(row=row, column=1).value
            code_boite = ws.cell(row=row, column=2).value
            temperature = ws.cell(row=row, column=3).value
            type_mesure = ws.cell(row=row, column=4).value

            if valeur_est_vide(espece) and valeur_est_vide(code_boite):
                continue

            # Température manquante
            if not valeur_est_vide(code_boite) and valeur_est_vide(temperature):
                temperatures_manquantes.append({
                    "fichier": excel_path.name,
                    "feuille": sheet_name,
                    "ligne_excel": row,
                    "espece": espece,
                    "code_boite": code_boite,
                    "temperature": temperature,
                    "type_mesure": type_mesure,
                })

            # Valeurs additionnées
            for col in range(5, ws.max_column + 1):
                valeur = ws.cell(row=row, column=col).value

                if isinstance(valeur, str) and REGEX_ADDITION.match(valeur):
                    mois = ws.cell(row=1, column=col).value
                    semaine = ws.cell(row=2, column=col).value
                    lettre_colonne = get_column_letter(col)

                    valeurs_additionnees.append({
                        "fichier": excel_path.name,
                        "feuille": sheet_name,
                        "ligne_excel": row,
                        "colonne": lettre_colonne,
                        "mois": mois,
                        "semaine": semaine,
                        "espece": espece,
                        "code_boite": code_boite,
                        "temperature": temperature,
                        "type_mesure": type_mesure,
                        "valeur_brute": valeur,
                        "valeur_calculee": calculer_addition(valeur),
                    })


valeurs_additionnees_df = pd.DataFrame(valeurs_additionnees)
temperatures_manquantes_df = pd.DataFrame(temperatures_manquantes)

display(Markdown("## Résumé"))
display(pd.DataFrame({
    "controle": ["Valeurs additionnées", "Températures manquantes"],
    "nombre": [len(valeurs_additionnees_df), len(temperatures_manquantes_df)]
}))

display(Markdown("## Valeurs additionnées détectées"))
if valeurs_additionnees_df.empty:
    display(Markdown("Aucune valeur additionnée détectée."))
else:
    display(valeurs_additionnees_df)

display(Markdown("## Températures manquantes détectées"))
if temperatures_manquantes_df.empty:
    display(Markdown("Aucune température manquante détectée."))
else:
    display(temperatures_manquantes_df)

## Résumé

,controle,nombre
0,Valeurs additionnées,0
1,Températures manquantes,3


## Valeurs additionnées détectées

Aucune valeur additionnée détectée.

## Températures manquantes détectées

,fichier,feuille,ligne_excel,espece,code_boite,temperature,type_mesure
0,Suivi_2025.xlsx,UMR_MARBEC,3,Rhizostoma pulmo,RPU-FBA-1.05,None,Nb polypes
1,Suivi_2025.xlsx,UMR_MARBEC,7,None,RPU-FBA-1.07,None,Nb polypes
2,Suivi_2025.xlsx,UMR_MARBEC,11,None,RPU-FBA-1.08,None,Nb polypes
